# 02 — Text Preprocessing

The goal of this step is to turn the raw question text into a cleaner and more consistent form before vectorization or modeling.  
At this stage, we stay focused on text quality: missing values, normalization, stopwords, and optional stemming or lemmatization.

In practice, this kind of preprocessing helps reduce noise, keeps similar forms of the same text closer together, and makes later modeling steps more stable and easier to compare.

In [2]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from bs4 import MarkupResemblesLocatorWarning

from project_bootstrap import setup_project

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

sns.set_theme(style="whitegrid")

In [3]:
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_CANDIDATE = NOTEBOOK_DIR if (NOTEBOOK_DIR / "project_bootstrap.py").exists() else NOTEBOOK_DIR.parent

if str(PROJECT_CANDIDATE) not in sys.path:
    sys.path.insert(0, str(PROJECT_CANDIDATE))

from project_bootstrap import setup_project

PROJECT_ROOT = setup_project()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

%load_ext autoreload
%autoreload 2

from text_preprocessing import (
    ensure_nltk_resources,
    normalize_text,
    preprocess_classic_ml,
)

In [4]:
ensure_nltk_resources()

## 1. Load data

We start by loading the training dataset and doing a quick structural check.  
The goal here is not to repeat the full EDA, but to make sure the expected text columns are present before preprocessing begins.

In [5]:
TRAIN_CANDIDATES = [
    DATA_DIR / "quora_question_pairs_train.csv",
    DATA_DIR / "quora_question_pairs_train.csv.zip",
]

train_path = next((p for p in TRAIN_CANDIDATES if p.exists()), None)
if train_path is None:
    raise FileNotFoundError(
        "Training file was not found. Expected one of: "
        + ", ".join(p.name for p in TRAIN_CANDIDATES)
    )

df = pd.read_csv(train_path)
print(f"Project root: {PROJECT_ROOT}")
print(f"Training file: {train_path.name}")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Project root: D:\Git\Quora-project
Training file: quora_question_pairs_train.csv.zip
Shape: (323432, 6)
Columns: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate']


The dataset was loaded correctly: it contains **323,432 rows** and **6 columns**, including both raw text columns and the binary target.  
At this stage, the structure looks consistent, so the data is ready for text cleanup.

## 2. Missing text values

The most important missing values here are in `question1` and `question2`.  
Since these are text fields, the safest preprocessing choice is to replace missing values with empty strings instead of dropping rows.

In [6]:
missing = (
    df[["question1", "question2"]]
      .isna()
      .sum()
      .rename("missing_count")
      .to_frame()
)

missing["missing_share"] = missing["missing_count"] / len(df)
missing

,missing_count,missing_share
question1,1,0.000003
question2,2,0.000006


In [7]:
df["question1"] = df["question1"].fillna("")
df["question2"] = df["question2"].fillna("")

df[["question1", "question2"]].isna().sum()

question1    0
question2    0
dtype: int64

Missing values in the text columns are almost absent: **1** row in `question1` and **2** rows in `question2`.  
After filling them with empty strings, both columns contain **0 missing values**, and the dataset size stays unchanged.

## 3. Class balance check

The target is slightly imbalanced, so it is useful to confirm that before moving on.  
Still, class balancing itself does not belong to text preprocessing, so here we only inspect it and keep the original distribution unchanged.

In [8]:
target_share = (
    df["is_duplicate"]
      .value_counts(normalize=True)
      .rename("share")
      .sort_index()
      .to_frame()
)

target_share["count"] = df["is_duplicate"].value_counts().sort_index()
target_share

,share,count
is_duplicate,,
0,0.630803,204022
1,0.369197,119410


The target distribution is moderately imbalanced: about **63.1%** non-duplicates and **36.9%** duplicates.  
This is worth keeping in mind for later evaluation, but it is not strong enough to justify any resampling inside preprocessing.

## 4. Minimal normalization

The first text version should stay close to the original meaning while removing obvious noise.  
For that, we apply a light normalization pipeline: lowercase, contraction expansion, number and symbol normalization, punctuation cleanup, HTML stripping, and whitespace normalization.

In [9]:
df["q1_norm"] = df["question1"].map(lambda x: normalize_text(x, lowercase=True))
df["q2_norm"] = df["question2"].map(lambda x: normalize_text(x, lowercase=True))

df[["question1", "q1_norm", "question2", "q2_norm"]].head(5)

,question1,q1_norm,question2,q2_norm
0,The Iliad and the Odyssey in the Greek culture?,the iliad and the odyssey in the greek culture,How do I prove that the pairs of three indepen...,how do i prove that the pairs of three indepen...
1,What is practical management and what is strat...,what is practical management and what is strat...,What are the practical aspects of strategic ma...,what are the practical aspects of strategic ma...
2,How useful is MakeUseOf Answers?,how useful is makeuseof answers,Is there any Q&A site that is not Yahoo answer...,is there any q a site that is not yahoo answer...
3,Which is the best place to reside in India and...,which is the best place to reside in india and...,Which ia the best place to visit in India?,which ia the best place to visit in india
4,Why do so many people ask questions on Quora t...,why do so many people ask questions on quora t...,Why don't many people posting questions on Quo...,why do not many people posting questions on qu...


The normalized text looks cleaner and more consistent: everything is lowercased, punctuation is removed, and the wording stays close to the original.  
From the preview, this version preserves the meaning of the questions while removing mostly superficial noise.

## 5. Stopword removal for a classic ML version

For count-based models such as Bag of Words or TF-IDF, it is often useful to remove very common function words.  
To keep this separate from the minimal normalized text, we create a second version specifically for classic ML workflows.

In [10]:
df["q1_classic"] = df["question1"].map(
    lambda x: preprocess_classic_ml(
        x,
        remove_stop_words=True,
        use_stemming=False,
        use_lemmatization=False,
    )
)

df["q2_classic"] = df["question2"].map(
    lambda x: preprocess_classic_ml(
        x,
        remove_stop_words=True,
        use_stemming=False,
        use_lemmatization=False,
    )
)

df[["q1_norm", "q1_classic", "q2_norm", "q2_classic"]].head(5)

,q1_norm,q1_classic,q2_norm,q2_classic
0,the iliad and the odyssey in the greek culture,iliad odyssey greek culture,how do i prove that the pairs of three indepen...,prove pairs three independent variables also i...
1,what is practical management and what is strat...,practical management strategic management,what are the practical aspects of strategic ma...,practical aspects strategic management
2,how useful is makeuseof answers,useful makeuseof answers,is there any q a site that is not yahoo answer...,q site yahoo answers hate speech allowed
3,which is the best place to reside in india and...,best place reside india,which ia the best place to visit in india,ia best place visit india
4,why do so many people ask questions on quora t...,many people ask questions quora easily answere...,why do not many people posting questions on qu...,many people posting questions quora check goog...


After stopword removal, the text becomes shorter and more compact, while the main content words remain.  
From the sample rows, this representation looks more suitable for count-based models, even though it is less natural to read than the normalized version.

## 6. Optional lemmatization and stemming

Both lemmatization and stemming can reduce vocabulary size, but they do it differently.  
Lemmatization is usually more conservative and readable, while stemming is more aggressive and can make the text look unnatural.

Instead of choosing one blindly, it is useful to compare a few examples side by side.

In [11]:
sample = df[["question1", "question2"]].head(5).copy()

sample["q1_stopwords_removed"] = sample["question1"].map(
    lambda x: preprocess_classic_ml(
        x,
        remove_stop_words=True,
        use_stemming=False,
        use_lemmatization=False,
    )
)

sample["q1_lemmatized"] = sample["question1"].map(
    lambda x: preprocess_classic_ml(
        x,
        remove_stop_words=True,
        use_stemming=False,
        use_lemmatization=True,
    )
)

sample["q1_stemmed"] = sample["question1"].map(
    lambda x: preprocess_classic_ml(
        x,
        remove_stop_words=True,
        use_stemming=True,
        use_lemmatization=False,
    )
)

sample[["question1", "q1_stopwords_removed", "q1_lemmatized", "q1_stemmed"]]

,question1,q1_stopwords_removed,q1_lemmatized,q1_stemmed
0,The Iliad and the Odyssey in the Greek culture?,iliad odyssey greek culture,iliad odyssey greek culture,iliad odyssey greek cultur
1,What is practical management and what is strat...,practical management strategic management,practical management strategic management,practic manag strateg manag
2,How useful is MakeUseOf Answers?,useful makeuseof answers,useful makeuseof answer,use makeuseof answer
3,Which is the best place to reside in India and...,best place reside india,best place reside india,best place resid india
4,Why do so many people ask questions on Quora t...,many people ask questions quora easily answere...,many people ask question quora easily answered...,mani peopl ask question quora easili answer nu...


In the sample rows, lemmatization stays close to the stopword-removed version, while stemming makes the text noticeably rougher and less readable.  
For this dataset, stemming looks too aggressive as a default choice, so it makes sense to keep it only as an optional experiment.

## 7. Save the preprocessed dataset

At the end of preprocessing, we keep the original raw questions and add the cleaned text columns.  
This makes it easier to inspect the output later and keeps the preprocessing step reusable for different downstream approaches.

In [12]:
keep_cols = [
    c for c in [
        "id",
        "qid1",
        "qid2",
        "question1",
        "question2",
        "is_duplicate",
        "q1_norm",
        "q2_norm",
        "q1_classic",
        "q2_classic",
    ]
    if c in df.columns
]

out_df = df[keep_cols].copy()

csv_path = PROCESSED_DIR / "quora_preprocessed.csv"
parquet_path = PROCESSED_DIR / "quora_preprocessed.parquet"

out_df.to_csv(csv_path, index=False)
out_df.to_parquet(parquet_path, index=False)

print(f"Saved CSV: {csv_path}")
print(f"Saved Parquet: {parquet_path}")
print(f"Final shape: {out_df.shape}")

Saved CSV: D:\Git\Quora-project\data\processed\quora_preprocessed.csv
Saved Parquet: D:\Git\Quora-project\data\processed\quora_preprocessed.parquet
Final shape: (323432, 10)


The preprocessing step produced a final dataset with **323,432 rows** and **10 columns** and saved it in both **CSV** and **Parquet** formats.  
The saved file now keeps the raw text together with two cleaned versions, which is enough for the next stage without mixing in engineered features.

## 8. Preprocessing summary

The preprocessing step stayed focused on cleaning and standardizing the raw question text.

Main outcomes:
- the dataset structure is valid and ready for further work;
- missing question texts were negligible and safely replaced with empty strings;
- the class distribution is moderately imbalanced, but no balancing was applied here;
- a lightly normalized text version was created for general use;
- a second, more compact version was created with stopword removal for classic ML;
- stemming looked too aggressive in the sample comparison, so it is not used by default.

Overall, the data is now cleaner, more consistent, and ready for feature extraction or vectorization.